In [1]:
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import KFold
from torch_geometric.data import HeteroData
from torch_geometric.nn import GCNConv, Linear, RGCNConv
import math

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

In [2]:
class DataLoader:
    def __init__(self, dataset='HMDAD'):
        self.dataset = dataset
        self.adj_path = f"{dataset}/mda.csv"
        self.mmi_path = f"{dataset}/mm_sim.txt"
        self.ddi_path = f"{dataset}/dd_sim.txt"
        self.m_embed_path = f"{dataset}/microbe_embeddings.csv"
        self.d_embed_path = f"{dataset}/disease_embeddings.csv"

    def load_data(self):
        mda = pd.read_csv(self.adj_path, index_col=0, sep=",")
        mda = mda.values.astype(np.int64)
        microbe_idx, disease_idx = np.where(mda == 1)
        edges = np.stack([microbe_idx, disease_idx], axis=1)

        mm_sim = np.loadtxt(self.mmi_path)
        dd_sim = np.loadtxt(self.ddi_path)
        print(f"Microbe similarity matrix shape: {mm_sim.shape}")
        print(f"Disease similarity matrix shape: {dd_sim.shape}")

        m_emb = pd.read_csv(self.m_embed_path, index_col=0).values
        d_emb = pd.read_csv(self.d_embed_path, index_col=0).values
        print(f"Microbe embedding matrix shape: {m_emb.shape}")
        print(f"Disease embedding matrix shape: {d_emb.shape}")

        return mda, edges, mm_sim, dd_sim, m_emb, d_emb

In [3]:
class GraphBuilder:
    def __init__(self, edges, mm_sim, dd_sim, m_emb, d_emb, d_k=15, m_k=25):
        self.edges = edges
        self.mm_sim = mm_sim
        self.dd_sim = dd_sim
        self.m_emb = m_emb
        self.d_emb = d_emb
        self.n_microbes = mm_sim.shape[0]
        self.n_diseases = dd_sim.shape[0]

        self.d_k = d_k
        self.m_k = m_k

    def build_graph_by_topk(self, sim_matrix, k):
        n_nodes = sim_matrix.shape[0]
        sim_matrix = sim_matrix.copy()
        np.fill_diagonal(sim_matrix, -1e9)

        k = min(k, n_nodes - 1) if n_nodes > 1 else 0
        if k <= 0:
            return np.zeros((0, 2), dtype=np.int64)

        idx = np.argpartition(-sim_matrix, kth=k, axis=1)[:, :k]
        rows = np.repeat(np.arange(n_nodes), k)
        cols = idx.reshape(-1)
        edges = np.stack([rows, cols], axis=1).astype(np.int64)
        edges = np.unique(edges, axis=0)
        edges = torch.LongTensor(edges)
        return edges

    def build_heterogeneous_graph(self):
        hetero_data = HeteroData()

        hetero_data['microbe'].x_sim = torch.from_numpy(self.mm_sim.astype(np.float32))
        hetero_data['disease'].x_sim = torch.from_numpy(self.dd_sim.astype(np.float32))
        hetero_data['microbe'].x_sem = torch.from_numpy(self.m_emb.astype(np.float32))
        hetero_data['disease'].x_sem = torch.from_numpy(self.d_emb.astype(np.float32))

        dd_edges = self.build_graph_by_topk(self.dd_sim, self.d_k)
        mm_edges = self.build_graph_by_topk(self.mm_sim, self.m_k)
        dd = torch.tensor(dd_edges.T, dtype=torch.long)
        mm = torch.tensor(mm_edges.T, dtype=torch.long)
        hetero_data['disease', 'similar', 'disease'].edge_index = dd
        hetero_data['microbe', 'similar', 'microbe'].edge_index = mm

        m_d_edges = torch.tensor(self.edges.T, dtype=torch.long)
        hetero_data['microbe', 'associated', 'disease'].edge_index = torch.LongTensor(m_d_edges)
        hetero_data['disease', 'rev_associated', 'microbe'].edge_index = torch.LongTensor(m_d_edges[[1, 0], :])

        return hetero_data

In [4]:
def info_nce(z1: torch.Tensor, z2: torch.Tensor, tau: float = 0.2) -> torch.Tensor:
    z1 = F.normalize(z1, dim=-1)
    z2 = F.normalize(z2, dim=-1)

    labels = torch.arange(z1.size(0), device=z1.device)

    logits12 = (z1 @ z2.t()) / tau
    loss12 = F.cross_entropy(logits12, labels)

    logits21 = (z2 @ z1.t()) / tau
    loss21 = F.cross_entropy(logits21, labels)

    return 0.5 * (loss12 + loss21)

In [5]:
class MDAPredictor(nn.Module):
    def __init__(self, in_m_sim, in_m_sem, in_d_sim, in_d_sem,hidden_dim=256, embed_dim=256, dropout=0.2,num_relations=4):
        super(MDAPredictor, self).__init__()

        self.m_sim_proj = nn.Sequential(
            Linear(in_m_sim, hidden_dim), nn.ReLU(), nn.Dropout(dropout), Linear(hidden_dim, hidden_dim)
        )
        self.m_sem_proj = nn.Sequential(
            Linear(in_m_sem, hidden_dim), nn.ReLU(), nn.Dropout(dropout), Linear(hidden_dim, hidden_dim)
        )
        self.d_sim_proj = nn.Sequential(
            Linear(in_d_sim, hidden_dim), nn.ReLU(), nn.Dropout(dropout), Linear(hidden_dim, hidden_dim)
        )
        self.d_sem_proj = nn.Sequential(
            Linear(in_d_sem, hidden_dim), nn.ReLU(), nn.Dropout(dropout), Linear(hidden_dim, hidden_dim)
        )

        self.gcn1 = GCNConv(hidden_dim, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, embed_dim)

        self.rgcn1 = RGCNConv(hidden_dim, hidden_dim, num_relations=num_relations)
        self.rgcn2 = RGCNConv(hidden_dim, embed_dim, num_relations=num_relations)

        self.cl_loss_m = nn.Sequential(nn.Linear(embed_dim, embed_dim), nn.ReLU(), Linear(embed_dim, embed_dim))
        self.cl_loss_d = nn.Sequential(nn.Linear(embed_dim, embed_dim), nn.ReLU(), Linear(embed_dim, embed_dim))

        self.m_fusion = nn.Sequential(nn.Linear(embed_dim * 2, embed_dim), nn.ReLU())
        self.d_fusion = nn.Sequential(nn.Linear(embed_dim * 2, embed_dim), nn.ReLU())

        self.W = nn.Parameter(torch.empty(embed_dim, embed_dim))
        nn.init.xavier_uniform_(self.W)
        self.P = nn.Parameter(torch.empty(embed_dim, 64))
        self.Q = nn.Parameter(torch.empty(embed_dim, 64))
        nn.init.xavier_uniform_(self.P); nn.init.xavier_uniform_(self.Q)

        self.dropout = dropout

    def homo_encode(self, data, microbe_x, disease_x):
        mm_edge = data['microbe', 'similar', 'microbe'].edge_index
        x_microbe = self.gcn1(microbe_x, mm_edge)
        x_microbe = F.relu(x_microbe)
        x_microbe = F.dropout(x_microbe, p=self.dropout, training=self.training)
        x_microbe = self.gcn2(x_microbe, mm_edge)

        dd_edge = data['disease', 'similar', 'disease'].edge_index
        x_disease = self.gcn1(disease_x, dd_edge)
        x_disease = F.relu(x_disease)
        x_disease = F.dropout(x_disease, p=self.dropout, training=self.training)
        x_disease = self.gcn2(x_disease, dd_edge)

        return {"microbe": x_microbe, "disease": x_disease}

    def hetero_encode(self, data, microbe_x, disease_x):
        data = data.clone()

        data["microbe"].x = microbe_x
        data["disease"].x = disease_x

        homo = data.to_homogeneous(node_attrs=["x"])
        x = homo.x
        edge_index = homo.edge_index
        edge_type = homo.edge_type

        x = self.rgcn1(x, edge_index, edge_type)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.rgcn2(x, edge_index, edge_type)

        node_type = homo.node_type
        microbe_type = data.node_types.index("microbe")
        disease_type = data.node_types.index("disease")

        z_microbe = x[node_type == microbe_type]
        z_disease = x[node_type == disease_type]

        return {"microbe": z_microbe, "disease": z_disease}

    def encode(self, data):
        m_sim = self.m_sim_proj(data["microbe"].x_sim)
        d_sim = self.d_sim_proj(data["disease"].x_sim)
        z_sim = self.homo_encode(data, m_sim, d_sim)

        m_sem = self.m_sem_proj(data["microbe"].x_sem)
        d_sem = self.d_sem_proj(data["disease"].x_sem)
        z_sem = self.hetero_encode(data, m_sem, d_sem)

        return z_sim, z_sem

    def decode(self, z_sim, z_sem):
        z_m = self.m_fusion(torch.cat([z_sim["microbe"], z_sem["microbe"]], dim=-1))
        z_d = self.d_fusion(torch.cat([z_sim["disease"], z_sem["disease"]], dim=-1))

        logits = (z_m @ self.P) @ (z_d @ self.Q).t()

        return logits

    def ori_cl_loss(self, z_sim, z_sem, tau):
        m1 = F.dropout(z_sim["microbe"], p=self.dropout, training=self.training)
        m2 = F.dropout(z_sem["microbe"], p=self.dropout, training=self.training)
        loss_m = info_nce(self.cl_loss_m(m1), self.cl_loss_m(m2), tau)

        d1 = F.dropout(z_sim["disease"], p=self.dropout, training=self.training)
        d2 = F.dropout(z_sem["disease"], p=self.dropout, training=self.training)
        loss_d = info_nce(self.cl_loss_d(d1), self.cl_loss_d(d2), tau)
        return loss_m + loss_d

    def neighbor_readout(self, z, edge_index, num_nodes):
        device = z.device
        edge_index = edge_index.to(device)

        row, col = edge_index

        assert row.max().item() < num_nodes
        assert col.max().item() < num_nodes

        deg = torch.zeros((num_nodes,), device=device, dtype=z.dtype)
        deg.index_add_(0, row, torch.ones_like(row, dtype=z.dtype, device=device))
        weight = 1.0 / torch.sqrt(deg[row] * deg[col] + 1e-6)

        out = torch.zeros((num_nodes, z.size(1)), device=device, dtype=z.dtype)
        out.index_add_(0, row, z[col] * weight.unsqueeze(-1))

        return out

    def cross_neighbor_readout(self, z_src, z_dst, edge_index, num_dst):
        device = z_dst.device
        edge_index = edge_index.to(device)

        src, dst = edge_index

        assert dst.max().item() < num_dst
        deg = torch.zeros((num_dst,), device=device, dtype=z_dst.dtype)
        deg.index_add_(0, dst, torch.ones_like(dst, dtype=z_dst.dtype))
        weight = 1.0 / torch.sqrt(deg[dst] + 1e-6)

        out = torch.zeros((num_dst, z_dst.size(1)), device=device, dtype=z_dst.dtype)
        out.index_add_(0, dst, z_src[src] * weight.unsqueeze(-1))

        return out

    def build_struct_repr(self, data, z_sim, z_sem):
        mm_edge = data['microbe', 'similar', 'microbe'].edge_index
        dd_edge = data['disease', 'similar', 'disease'].edge_index
        # print(z_sim['microbe'].size(0),z_sim['disease'].size(0))    # 292，39

        s_m_sim = self.neighbor_readout(
            z_sim['microbe'], mm_edge, z_sim['microbe'].size(0)
        )
        s_m_sim = s_m_sim - z_sim['microbe']    # new
        s_d_sim = self.neighbor_readout(
            z_sim['disease'], dd_edge, z_sim['disease'].size(0)
        )
        s_d_sim = s_d_sim - z_sim['disease']    # new

        # SEM
        # microbe ← disease
        s_m_sem = self.cross_neighbor_readout(
            z_sem['disease'],
            z_sem['microbe'],
            data['disease', 'rev_associated', 'microbe'].edge_index,
            z_sem['microbe'].size(0)
        )
        s_m_sem = s_m_sem - z_sem['microbe']    # new

        # disease ← microbe
        s_d_sem = self.cross_neighbor_readout(
            z_sem['microbe'],
            z_sem['disease'],
            data['microbe', 'associated', 'disease'].edge_index,
            z_sem['disease'].size(0)
        )
        s_d_sem = s_d_sem - z_sem['disease']    # new

        return {
            'microbe': (s_m_sim, s_m_sem),
            'disease': (s_d_sim, s_d_sem)
        }
    def struct_cl_loss(self, data, z_sim, z_sem, tau):
        struct = self.build_struct_repr(data, z_sim, z_sem)

        s_m_sim, s_m_sem = struct['microbe']
        s_m_sim = self.cl_loss_m(F.dropout(s_m_sim, p=self.dropout, training=self.training))
        s_m_sem = self.cl_loss_m(F.dropout(s_m_sem, p=self.dropout, training=self.training))
        loss_m = info_nce(s_m_sim, s_m_sem, tau)

        s_d_sim, s_d_sem = struct['disease']
        s_d_sim = self.cl_loss_d(F.dropout(s_d_sim, p=self.dropout, training=self.training))
        s_d_sem = self.cl_loss_d(F.dropout(s_d_sem, p=self.dropout, training=self.training))
        loss_d = info_nce(s_d_sim, s_d_sem, tau)

        return loss_m + loss_d

In [6]:
def build_A_from_edges(pos_edges: np.ndarray, Nm: int, Nd: int) -> torch.Tensor:
    A = torch.zeros((Nm, Nd), dtype=torch.float32)
    if pos_edges.size > 0:
        A[pos_edges[:, 0], pos_edges[:, 1]] = 1.0
    return A

def sample_fixed_negatives(A_full: torch.Tensor, num_negs: int, seed=42):
    rng = np.random.default_rng(seed)
    A_np = A_full if isinstance(A_full, np.ndarray) else A_full.cpu().numpy()
    neg_coords = np.argwhere(A_np == 0)
    if len(neg_coords) == 0:
        return np.zeros((0, 2), dtype=np.int64)
    choose = rng.choice(len(neg_coords), size=min(num_negs, len(neg_coords)), replace=False)
    return neg_coords[choose].astype(np.int64)

def prepare_data_splits(mda, n_splits, val_ratio=0.1, seed=42):
    pos_edges = np.argwhere(mda == 1)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits  = []

    for fold, (train_val_idx, test_idx) in enumerate(kf.split(pos_edges)):
        test_pos = pos_edges[test_idx]
        train_val_pos = pos_edges[train_val_idx]
        idx = np.arange(len(train_val_pos))
        rng = np.random.default_rng(seed +  fold)
        rng.shuffle(idx)

        n_val = int(len(idx) * val_ratio)
        val_pos = train_val_pos[idx[:n_val]]
        train_pos = train_val_pos[idx[n_val:]]

        splits.append({
            'train_pos': train_pos,
            'val_pos': val_pos,
            'test_pos': test_pos,
        })

    return splits

In [7]:
class Trainer:
    def __init__(self, model, alpha, device='cuda'):
        self.model = model.to(device)
        self.alpha = alpha
        self.device = device
        self.optimizer = torch.optim.Adam(
            model.parameters(), lr=0.001, weight_decay=1e-5
        )

    def train_epoch(self, data, A_train, device, lam_cl, tau1, tau2, epoch):
        self.model.train()
        data = data.to(device)
        A_train = A_train.to(device)
        self.optimizer.zero_grad()

        z_sim, z_sem = self.model.encode(data)
        logits = self.model.decode(z_sim, z_sem)
        cl_loss1 = self.model.ori_cl_loss(z_sim, z_sem, tau1)
        cl_loss2 = self.model.struct_cl_loss(data,z_sim, z_sem, tau2)

        ratio = (A_train.numel() - A_train.sum()) / A_train.sum().clamp(min=1)
        pos_weight = torch.sqrt(ratio)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        loss_bce = loss_fn(logits, A_train)

        if epoch < 50:
            loss = loss_bce + lam_cl * self.alpha * cl_loss1 + lam_cl * (1 - self.alpha) * cl_loss2
        else:
            loss = loss_bce

        loss.backward()
        self.optimizer.step()

        return float(loss.item()), float(loss_bce.item()), float(cl_loss1.item()), float(cl_loss2.item())

    @torch.no_grad()
    def evaluate(self, data, pos_edges, neg_edges):
        self.model.eval()
        data.to(self.device)

        z_sim, z_sem = self.model.encode(data)
        logits = self.model.decode(z_sim, z_sem)

        pos_scores = torch.sigmoid(logits[pos_edges[:, 0], pos_edges[:, 1]]).cpu().numpy()
        neg_scores = torch.sigmoid(logits[neg_edges[:, 0], neg_edges[:, 1]]).cpu().numpy()

        y = np.concatenate([np.ones_like(pos_scores), np.zeros_like(neg_scores)])
        s = np.concatenate([pos_scores, neg_scores])
        auc = roc_auc_score(y, s) if len(np.unique(y)) == 2 else 0.5
        ap  = average_precision_score(y, s)
        return auc, ap

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("=== Loading Data ===")
datasets = ['Disbiome', 'HMDAD']
data_loader = DataLoader(dataset=datasets[1])
mda, edges_md, mm_sim, dd_sim, m_emb, d_emb = data_loader.load_data()

print("=== Splitting Positive Edges ===")
splits = prepare_data_splits(mda, n_splits=5, val_ratio=0.1, seed=42)

Using device: cuda
=== Loading Data ===
Microbe similarity matrix shape: (292, 292)
Disease similarity matrix shape: (39, 39)
Microbe embedding matrix shape: (292, 1536)
Disease embedding matrix shape: (39, 1536)
=== Splitting Positive Edges ===


In [9]:
alpha = 0.8
tau1 = 0.2
tau2 = 0.6
lam_cl = 0.3
print("=== Building Graphs ===")
graph_builder = GraphBuilder(edges_md, mm_sim, dd_sim, m_emb, d_emb, m_k=25,d_k=15)
hetero_data = graph_builder.build_heterogeneous_graph()

=== Building Graphs ===


In [12]:
print("=== Starting Cross-Validation  ===")
eval_neg_ratio = 1.0
n_microbes = mda.shape[0]
n_diseases = mda.shape[1]

results = []

set_seed(42)
def cosine_anneal(start, end, t):
    return end + 0.5 * (start - end) * (1 + math.cos(math.pi * t))

for fold, split in enumerate(splits):
    print(f"\n--- Fold {fold + 1} ---")


    train_pos = split['train_pos']
    val_pos = split['val_pos']
    test_pos = split['test_pos']

    A_train = build_A_from_edges(train_pos, n_microbes, n_diseases)
    A_full = build_A_from_edges(edges_md, n_microbes, n_diseases)

    num_test_negs = int(len(test_pos) * eval_neg_ratio)
    num_val_negs = int(len(val_pos) * eval_neg_ratio)

    val_neg = sample_fixed_negatives(A_full, num_val_negs, seed=42+1 +  fold*10)
    test_neg = sample_fixed_negatives(A_full, num_test_negs, seed=42+2 +  fold*10)


    in_m_sim = hetero_data["microbe"].x_sim.size(-1)
    in_m_sem = hetero_data["microbe"].x_sem.size(-1)
    in_d_sim = hetero_data["disease"].x_sim.size(-1)
    in_d_sem = hetero_data["disease"].x_sem.size(-1)

    model = MDAPredictor(
        in_m_sim, in_m_sem, in_d_sim, in_d_sem,
        hidden_dim=256,
        embed_dim=256,
        dropout=0.2,
        num_relations=4
    )

    trainer = Trainer(model, alpha, device=device)

    best_val = 0
    patience = 20
    patience_counter = 0
    best_state = None
    best_results = {'auc': 0, 'aupr': 0}

    max_epochs = 250
    for epoch in range(max_epochs):
        loss, bce_loss, cl_loss1, cl_loss2 = trainer.train_epoch(
            hetero_data, A_train, device, lam_cl=lam_cl, tau1=tau1, tau2=tau2, epoch=epoch
        )

        if (epoch + 1) % 10 == 0:
            val_auc, val_aupr = trainer.evaluate(
                hetero_data, val_pos, val_neg
            )
            print(f"Epoch {epoch+1:3d} | Loss: {loss:.4f} (BCE: {bce_loss:.4f}, CL1: {cl_loss1:.4f}),CL2: {cl_loss2:.4f}) \n "
                  f"Val AUC: {val_auc:.4f} | AUPR: {val_aupr:.4f}")

            if val_auc > best_val:
                best_val = val_auc
                patience_counter = 0
                best_results = {'auc': val_auc, 'aupr': val_aupr}
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                patience_counter += 1

            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    test_auc, test_aupr = trainer.evaluate(hetero_data, test_pos, test_neg)
    best_results.update({'test_auc': test_auc, 'test_aupr': test_aupr})
    results.append(best_results)
    print(f"Fold {fold+1} Best Results - Val AUC: {best_results['auc']:.4f}\n"
          f"Test AUC: {test_auc:.4f}, AUPR: {test_aupr:.4f}")

=== Starting Cross-Validation  ===

--- Fold 1 ---
Epoch  10 | Loss: 3.3377 (BCE: 0.5358, CL1: 9.3389),CL2: 9.3424) 
 Val AUC: 0.5833 | AUPR: 0.6357
Epoch  20 | Loss: 3.1967 (BCE: 0.3982, CL1: 9.3268),CL2: 9.3337) 
 Val AUC: 0.8380 | AUPR: 0.8737
Epoch  30 | Loss: 3.0644 (BCE: 0.2905, CL1: 9.2301),CL2: 9.3126) 
 Val AUC: 0.9097 | AUPR: 0.9157
Epoch  40 | Loss: 3.0939 (BCE: 0.2597, CL1: 9.4888),CL2: 9.2813) 
 Val AUC: 0.9290 | AUPR: 0.9289
Epoch  50 | Loss: 2.9192 (BCE: 0.2332, CL1: 8.8884),CL2: 9.2136) 
 Val AUC: 0.9383 | AUPR: 0.9297
Epoch  60 | Loss: 0.2029 (BCE: 0.2029, CL1: 8.8772),CL2: 9.2014) 
 Val AUC: 0.9452 | AUPR: 0.9274
Epoch  70 | Loss: 0.1928 (BCE: 0.1928, CL1: 8.9682),CL2: 9.2017) 
 Val AUC: 0.9444 | AUPR: 0.9308
Epoch  80 | Loss: 0.1745 (BCE: 0.1745, CL1: 9.0537),CL2: 9.1677) 
 Val AUC: 0.9583 | AUPR: 0.9589
Epoch  90 | Loss: 0.1476 (BCE: 0.1476, CL1: 9.2090),CL2: 9.1913) 
 Val AUC: 0.9637 | AUPR: 0.9636
Epoch 100 | Loss: 0.1337 (BCE: 0.1337, CL1: 9.2267),CL2: 9.2060) 
 

In [11]:
print("\n=== Final Results ===")
avg_auc  = np.mean([r['auc']  for r in results])
avg_aupr = np.mean([r['aupr'] for r in results])
std_auc  = np.std([r['auc']  for r in results])
std_aupr = np.std([r['aupr'] for r in results])

print(f"Val   AUC:  {avg_auc:.4f} ± {std_auc:.4f}")
print(f"Val  AUPR: {avg_aupr:.4f} ± {std_aupr:.4f}")

auc_mean = np.mean([r['test_auc'] for r in results])
auc_std = np.std([r['test_auc'] for r in results])
aupr_mean = np.mean([r['test_aupr'] for r in results])
aupr_std = np.std([r['test_aupr'] for r in results])
print("Test AUC\tTest AUPR")

print(f"{auc_mean:.4f} ± {auc_std:.4f}\t"
      f"{aupr_mean:.4f} ± {aupr_std:.4f}\t")


=== Final Results ===
Val   AUC:  0.9898 ± 0.0049
Val  AUPR: 0.9893 ± 0.0051
Test AUC	Test AUPR
0.9828 ± 0.0076	0.9785 ± 0.0113	
